In [2]:
import sys
import os
import importlib
import pandas as pd
from pathlib import Path

# Xác định thư mục gốc dự án
PROJECT_ROOT = Path.cwd().parent

# Thêm thư mục gốc vào Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import Game Engine
from game.game_engine import GameEngine

# Import các Agent
from agents.random_agent import RandomAgent
from agents.heuristic_agent import HeuristicAgent
from agents.minimax_agent import MinimaxAgent
from agents.alphabeta_agent import AlphaBetaAgent

# Reload MCTS
import agents.mcts_agent as mcts_module
importlib.reload(mcts_module)

from agents.mcts_agent import MCTSAgent

print("Import thành công!")
print("Project root:", PROJECT_ROOT)

Import thành công!
Project root: D:\TTNTNC\TicTacToe_AI


In [3]:
agent_factories = {
    "MCTS": lambda: MCTSAgent(
        simulations=500,
        exploration=2**0.5
    ),

    "Random": lambda: RandomAgent(),

    "Heuristic": lambda: HeuristicAgent(),

    "Minimax": lambda: MinimaxAgent(depth=4),

    "Alpha-Beta": lambda: AlphaBetaAgent(depth=4)
}

print("Danh sách Agent:")

for name in agent_factories:
    print("-", name)

Danh sách Agent:
- MCTS
- Random
- Heuristic
- Minimax
- Alpha-Beta


In [4]:
agent_factories = {
    "MCTS": lambda: MCTSAgent(
        simulations=500,
        exploration=2**0.5
    ),

    "Random": lambda: RandomAgent(),

    "Heuristic": lambda: HeuristicAgent(),

    "Minimax": lambda: MinimaxAgent(depth=4),

    "Alpha-Beta": lambda: AlphaBetaAgent(depth=4)
}

print("Danh sách Agent:")

for name in agent_factories:
    print("-", name)

Danh sách Agent:
- MCTS
- Random
- Heuristic
- Minimax
- Alpha-Beta


In [5]:
def get_winner(board, win_length=4):
    board_size = len(board)

    directions = [
        (0, 1),    # Ngang
        (1, 0),    # Dọc
        (1, 1),    # Chéo xuống
        (1, -1)    # Chéo lên
    ]

    for r in range(board_size):
        for c in range(board_size):

            player = board[r][c]

            if player == 0:
                continue

            for dr, dc in directions:

                cells = []

                for i in range(win_length):
                    nr = r + i * dr
                    nc = c + i * dc

                    if not (0 <= nr < board_size and
                            0 <= nc < board_size):
                        break

                    cells.append(board[nr][nc])

                if len(cells) == win_length:
                    if all(value == player for value in cells):
                        return player

    return 0


print("Hàm kiểm tra người thắng đã sẵn sàng!")

Hàm kiểm tra người thắng đã sẵn sàng!


In [6]:
def play_match(x_name, o_name):
    # Khởi tạo Game Engine
    game = GameEngine()

    # Khởi tạo hai Agent
    agent_x = agent_factories[x_name]()
    agent_o = agent_factories[o_name]()

    agents = {
        1: agent_x,
        -1: agent_o
    }

    move_count = 0

    while not game.is_terminal():

        # Xác định người chơi hiện tại
        current_player = game.state.current_player

        # Lấy Agent tương ứng
        current_agent = agents[current_player]

        # Agent lựa chọn hành động
        action = current_agent.choose_action(game)

        # Kiểm tra nước đi hợp lệ
        if action not in game.legal_actions():
            raise ValueError(
                f"{current_agent.name} chọn nước đi không hợp lệ: {action}"
            )

        # Thực hiện nước đi
        game.step(action)

        move_count += 1

    # Xác định người chiến thắng
    winner = get_winner(
        game.state.board,
        game.win_length
    )

    if winner == 1:
        winner_name = x_name

    elif winner == -1:
        winner_name = o_name

    else:
        winner_name = "Draw"

    return {
        "X_Agent": x_name,
        "O_Agent": o_name,
        "Winner": winner_name,
        "Moves": move_count
    }

In [7]:
test_result = play_match("MCTS", "Random")

print("KẾT QUẢ TRẬN ĐẤU")
print("-------------------------")
print("X:", test_result["X_Agent"])
print("O:", test_result["O_Agent"])
print("Winner:", test_result["Winner"])
print("Moves:", test_result["Moves"])

KẾT QUẢ TRẬN ĐẤU
-------------------------
X: MCTS
O: Random
Winner: Draw
Moves: 16


In [11]:
opponents = [
    "Random",
    "Heuristic",
    "Minimax",
    "Alpha-Beta"
]

match_results = []

for opponent in opponents:

    print(f"\nĐang thi đấu: MCTS vs {opponent}")

    for match_number in range(4):

        # Đổi bên sau mỗi trận
        if match_number % 2 == 0:
            x_name = "MCTS"
            o_name = opponent
        else:
            x_name = opponent
            o_name = "MCTS"

        result = play_match(x_name, o_name)

        # Xác định kết quả từ góc nhìn MCTS
        if result["Winner"] == "MCTS":
            mcts_result = "Win"

        elif result["Winner"] == "Draw":
            mcts_result = "Draw"

        else:
            mcts_result = "Loss"

        result["Opponent"] = opponent
        result["MCTS_Result"] = mcts_result
        result["Match_Number"] = match_number + 1

        match_results.append(result)

        print(
            f"Trận {match_number + 1}: "
            f"{mcts_result} | "
            f"Số lượt: {result['Moves']}"
        )

print("\nHOÀN THÀNH TOURNAMENT!")
print("Tổng số trận:", len(match_results))


Đang thi đấu: MCTS vs Random
Trận 1: Win | Số lượt: 7
Trận 2: Win | Số lượt: 8
Trận 3: Win | Số lượt: 7
Trận 4: Win | Số lượt: 10

Đang thi đấu: MCTS vs Heuristic
Trận 1: Draw | Số lượt: 16
Trận 2: Draw | Số lượt: 16
Trận 3: Draw | Số lượt: 16
Trận 4: Draw | Số lượt: 16

Đang thi đấu: MCTS vs Minimax


TypeError: MinimaxAgent() takes no arguments